# 02 · Vessel segmentation, vessel centreline graph & candidate embolus ROIs
Implements the *vessel-topology representation*: the vascular tree is a graph **G = (V, E)** where each node `v_i` is a vessel segment (between junctions) and each edge `e_ij` an anatomical connection.

Per-segment features (14-d): length, mean/max/min radius, mean/std/min/10th-percentile centreline intensity, **filling-defect (dip) score** relative to neighbouring segments, neighbour reference intensity, straightness, and normalised (z, y, x) position.

The top-K segments ranked by `dip × radius` become **candidate embolus ROIs**; graph hop distances between them feed the topology attention in the network.

> **Honest caveat.** The segmentation below is a *classical intensity/morphology pseudo-segmenter* (lung hull → bright structures → closing). It is good enough for a self-contained pipeline and for weak auxiliary supervision, but for a journal submission you should replace `segment_vessels` with a validated pulmonary-artery/vein segmenter (e.g. TotalSegmentator or an nnU-Net) and report its accuracy. The overlay cell at the bottom lets you visually verify it on your data.

In [ ]:
%run ./00_common.ipynb

In [ ]:
import numpy as np
from scipy import ndimage as ndi
from skimage.morphology import skeletonize
from skimage.filters import threshold_otsu
import networkx as nx

S26 = np.ones((3, 3, 3), bool)

def lung_and_hull(vol, air_thr=0.04, min_frac=0.005):
    """vol: (D,H,W) float in [0,1]. Returns lung-air mask and a hull (lung + enclosed vessels)."""
    air = vol <= air_thr
    lab, n = ndi.label(air, structure=ndi.generate_binary_structure(3, 1))
    if n == 0:
        return np.zeros_like(air), np.zeros_like(air)
    sizes = ndi.sum(air, lab, index=np.arange(1, n + 1))
    border = np.zeros(vol.shape, bool)
    border[:, 0, :] = border[:, -1, :] = border[:, :, 0] = border[:, :, -1] = True
    touch = np.unique(lab[border & air])                     # components that reach the in-plane border = outside air
    keep = [i + 1 for i, s in enumerate(sizes) if (i + 1) not in touch and s > min_frac * vol.size]
    lung = np.isin(lab, keep)
    hull = ndi.binary_closing(lung, structure=ndi.generate_binary_structure(3, 1), iterations=3)
    hull = np.stack([ndi.binary_fill_holes(s) for s in hull])   # vessels inside lung = holes in each slice
    return lung, hull

def segment_vessels(vol, hilum_dilate=4, min_size=30, thr=None):
    """Classical pseudo-vessel segmentation (intensity inside dilated lung hull). Replace with a learned
    vessel/artery segmenter (e.g. TotalSegmentator, nnU-Net) for the final manuscript."""
    lung, hull = lung_and_hull(vol)
    if hull.sum() == 0:
        return np.zeros(vol.shape, np.uint8), lung
    roi = ndi.binary_dilation(hull, structure=ndi.generate_binary_structure(3, 1), iterations=hilum_dilate)
    cand = roi & ~lung & (vol > 0.04)
    if thr is None:
        vals = vol[cand]
        thr = float(np.clip(threshold_otsu(vals), 0.30, 0.55)) if vals.size > 100 else 0.40
    v = cand & (vol > thr)
    v = ndi.binary_opening(v, structure=ndi.generate_binary_structure(3, 1))
    # closing bridges filling defects (emboli are darker than contrast) so they stay *inside* the vessel mask
    v = ndi.binary_closing(v, structure=ndi.generate_binary_structure(3, 1), iterations=3)
    lab, n = ndi.label(v, structure=S26)
    if n:
        sizes = ndi.sum(v, lab, index=np.arange(1, n + 1))
        v = np.isin(lab, [i + 1 for i, s in enumerate(sizes) if s >= min_size])
    return v.astype(np.uint8), lung

def find_junctions(sk):
    """A skeleton voxel is a junction if its 26-neighbours form >=3 separate connected groups
    (plain neighbour counting over-fires on diagonal runs)."""
    pad = np.pad(sk, 1)
    junc = np.zeros(sk.shape, bool)
    for z, y, x in np.argwhere(sk):
        p = pad[z:z + 3, y:y + 3, x:x + 3].copy(); p[1, 1, 1] = False
        if p.sum() >= 3 and ndi.label(p, structure=S26)[1] >= 3:
            junc[z, y, x] = True
    return junc

NF = 14  # node-feature dimension

def vessel_graph_candidates(mask, vol, K=32, max_hop=3, min_seg=3):
    """Skeletonise -> split at junctions -> segment graph -> score segments -> top-K candidate ROIs."""
    D, H, W = mask.shape
    out = dict(centers=np.zeros((K, 3), np.float32), feats=np.zeros((K, NF), np.float32),
               hops=np.full((K, K), max_hop + 1, np.uint8), score=np.zeros(K, np.float32),
               valid=np.zeros(K, bool))
    if mask.sum() < 50:
        out['valid'][0] = True; out['hops'][0, 0] = 0
        return out, nx.Graph()
    sk = skeletonize(mask > 0).astype(bool)
    junction = find_junctions(sk)
    junc_d = ndi.binary_dilation(junction, structure=S26)
    seg_vox = sk & ~junc_d
    seg_lab, n = ndi.label(seg_vox, structure=S26)
    dist = ndi.distance_transform_edt(mask > 0)
    vs = ndi.uniform_filter(vol.astype(np.float32), 3)
    coords = np.argwhere(seg_lab > 0); labs = seg_lab[seg_lab > 0]
    order = np.argsort(labs); coords, labs = coords[order], labs[order]
    uniq, starts = np.unique(labs, return_index=True)
    groups = np.split(coords, starts[1:])
    segs = {}
    for l, c in zip(uniq, groups):
        if len(c) < min_seg: continue
        I = vs[c[:, 0], c[:, 1], c[:, 2]]; R = dist[c[:, 0], c[:, 1], c[:, 2]]
        mid = c[np.argmin(((c - c.mean(0)) ** 2).sum(1))]
        chord = np.linalg.norm(c[0] - c[-1]) if len(c) > 1 else 0.
        segs[int(l)] = dict(c=c, I=I, R=R, mid=mid, chord=chord)
    G = nx.Graph(); G.add_nodes_from(segs)
    jl, jn = ndi.label(junc_d & sk, structure=S26)
    for j in range(1, jn + 1):
        ring = ndi.binary_dilation(jl == j, structure=S26, iterations=2)
        touching = [int(x) for x in np.unique(seg_lab[ring]) if x in segs]
        G.add_edges_from([(a, b) for i, a in enumerate(touching) for b in touching[i + 1:]])
    if len(segs) == 0:
        out['valid'][0] = True; out['hops'][0, 0] = 0
        return out, G
    glob_med = np.median([s['I'].mean() for s in segs.values()])
    rows = []
    for l, s in segs.items():
        nbrs = [segs[m]['I'].mean() for m in G.neighbors(l)]
        ref = np.median(nbrs) if nbrs else glob_med
        dip = max(0., ref - s['I'].min())                 # filling-defect cue: segment darker than neighbours
        n = len(s['c'])
        z, y, x = s['mid']
        f = [min(n / 40., 2), s['R'].mean() / 8., s['R'].max() / 8., s['R'].min() / 8.,
             s['I'].mean(), s['I'].std() * 4, s['I'].min(), np.percentile(s['I'], 10),
             dip * 4, ref, min(s['chord'] / max(n, 1), 1.), z / (D - 1) * 2 - 1, y / (H - 1) * 2 - 1, x / (W - 1) * 2 - 1]
        score = dip * (0.5 + s['R'].mean() / 8.)
        rows.append((l, np.array(f, np.float32), score, s['mid']))
    rows.sort(key=lambda r: -r[2]); rows = rows[:K]
    ids = [r[0] for r in rows]
    for i, (l, f, sc, mid) in enumerate(rows):
        out['feats'][i] = f; out['score'][i] = sc; out['valid'][i] = True
        out['centers'][i] = [mid[2] / (W - 1) * 2 - 1, mid[1] / (H - 1) * 2 - 1, mid[0] / (D - 1) * 2 - 1]  # (x,y,z)
    for i, l in enumerate(ids):
        sp = nx.single_source_shortest_path_length(G, l, cutoff=max_hop)
        for j, m in enumerate(ids):
            out['hops'][i, j] = sp.get(m, max_hop + 1)
    return out, G

In [ ]:
from joblib import Parallel, delayed
sp = load_splits(); all_ids = sp["train"] + sp["val"] + sp["test"]

def process(sid):
    out_f = f"{cfg.vessel_dir}/{sid}.npz"
    if os.path.exists(out_f): return sid, True
    try:
        vol = np.load(f"{cfg.vol_dir}/{sid}.npz")["vol"].astype(np.float32)
        ves, _ = segment_vessels(vol)
        o, _ = vessel_graph_candidates(ves, vol, K=cfg.n_roi, max_hop=cfg.max_hop)
        np.savez_compressed(out_f, vessel=ves, roi_centers=o["centers"], roi_feats=o["feats"], roi_hops=o["hops"],
                            roi_score=o["score"], roi_valid=o["valid"])
        return sid, True
    except Exception as e:
        print("FAILED", sid, repr(e)); return sid, False

# smoke test on a few studies first, then everything
res = Parallel(n_jobs=-1 if not IS_KAGGLE else 4, verbose=5)(delayed(process)(s) for s in all_ids)
print("ok:", sum(r[1] for r in res), "/", len(res))

### Diagnostics
* **Vessel volume sanity** (voxels per study).
* **Candidate recall (z-level proxy)**: among PE-positive studies, how often does at least one *valid* candidate ROI lie within ±1 slice of a PE-labelled slice? This bounds what the topology-attention stage can recover; if it is low, loosen `cfg.cand_thr` or increase `cfg.n_roi`.

In [ ]:
rows = []
for sid in sp["train"][:300]:
    a = np.load(f"{cfg.vol_dir}/{sid}.npz"); b = np.load(f"{cfg.vessel_dir}/{sid}.npz")
    sl = a["slice_lab"]; hit = np.nan
    if a["y"][0] == 1 and not np.isnan(sl).any() and sl.sum() > 0:
        Dn = len(sl); zi = np.round((b["roi_centers"][b["roi_valid"], 2] + 1) / 2 * (Dn - 1)).astype(int)
        pe_z = np.where(sl > 0)[0]; hit = float(any(np.abs(zi[:, None] - pe_z[None]).min(1) <= 1)) if len(zi) else 0.
    rows.append(dict(sid=sid, vessel_vox=int(b["vessel"].sum()), n_valid=int(b["roi_valid"].sum()), pe=a["y"][0], cand_hit=hit))
import pandas as pd
df = pd.DataFrame(rows); print(df.describe().T[["mean", "min", "50%", "max"]])
print("candidate recall (PE+ studies):", np.nanmean(df.cand_hit).round(3))

In [ ]:
import matplotlib.pyplot as plt
sid = sp["train"][1]; a = np.load(f"{cfg.vol_dir}/{sid}.npz"); b = np.load(f"{cfg.vessel_dir}/{sid}.npz")
v, m = a["vol"].astype(np.float32), b["vessel"]; Dn = v.shape[0]
fig, ax = plt.subplots(2, 4, figsize=(15, 7.5))
for i, z in enumerate(np.linspace(10, Dn - 10, 4).astype(int)):
    ax[0, i].imshow(v[z], cmap="gray", vmin=0, vmax=1); ax[0, i].set_title(f"z={z}")
    ax[1, i].imshow(v[z], cmap="gray", vmin=0, vmax=1); ax[1, i].imshow(np.ma.masked_where(m[z] == 0, m[z]), cmap="autumn", alpha=.6)
    c = b["roi_centers"][b["roi_valid"]]; zz = (c[:, 2] + 1) / 2 * (Dn - 1); near = np.abs(zz - z) < 2
    ax[1, i].scatter((c[near, 0] + 1) / 2 * (v.shape[2] - 1), (c[near, 1] + 1) / 2 * (v.shape[1] - 1), s=40, facecolors="none", edgecolors="cyan")
    for r in (0, 1): ax[r, i].axis("off")
ax[0, 0].set_ylabel("CT"); plt.suptitle("Row 2: pseudo-vessel mask (orange) and candidate ROIs (cyan rings)"); plt.show()